In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# Ethnicity
# To download original dataset go to - 
# https://www.ons.gov.uk/datasets/TS021/editions/2021/versions/3
ethnicity_csv = r"N:\0_Leads\2424_Place+Portrait\WORKING\Data_Prep\Raw Data\CSV\TS021-2021-3.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [3]:
lsoa2021_shapefile = r"N:\0_Leads\2424_Place+Portrait\WORKING\Data_Prep\Raw Data\Boundaries\LAD_DEC_2022_UK_BFE_V2.shp"


## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [4]:
threshold_value = 2.5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [5]:
# Create Dataframe
lsoa_ethnicity_df = pd.read_csv(ethnicity_csv)
lsoa_ethnicity_df.head()

,Lower tier local authorities Code,Lower tier local authorities,Ethnic group (20 categories) Code,Ethnic group (20 categories),Observation
0,E06000001,Hartlepool,-8,Does not apply,0
1,E06000001,Hartlepool,1,"Asian, Asian British or Asian Welsh: Bangladeshi",278
2,E06000001,Hartlepool,2,"Asian, Asian British or Asian Welsh: Chinese",217
3,E06000001,Hartlepool,3,"Asian, Asian British or Asian Welsh: Indian",335
4,E06000001,Hartlepool,4,"Asian, Asian British or Asian Welsh: Pakistani",297


In [6]:
#create pivot table
lsoa_ethincity_df_P = pd.pivot_table(lsoa_ethnicity_df, values='Observation', index=['Lower tier local authorities Code'], columns=['Ethnic group (20 categories)'], aggfunc=np.sum)
lsoa_ethincity_df_P = lsoa_ethincity_df_P.reset_index()

#drop columns
lsoa_ethincity_df_P.drop(['Does not apply'], 1, inplace=True)

# Dictionary for renaming columns with corrected keys
column_rename_map = {
    "Asian, Asian British or Asian Welsh: Bangladeshi": "asian_bangladeshi",
    "Asian, Asian British or Asian Welsh: Chinese": "asian_chinese",
    "Asian, Asian British or Asian Welsh: Indian": "asian_indian",
    "Asian, Asian British or Asian Welsh: Pakistani": "asian_pakistani",
    "Asian, Asian British or Asian Welsh: Other Asian": "asian_other",
    
    "Black, Black British, Black Welsh, Caribbean or African: African": "black_african",
    "Black, Black British, Black Welsh, Caribbean or African: Caribbean": "black_caribbean",
    "Black, Black British, Black Welsh, Caribbean or African: Other Black": "black_other",
    
    "Mixed or Multiple ethnic groups: Other Mixed or Multiple ethnic groups": "mixed_or_multiple_other",
    "Mixed or Multiple ethnic groups: White and Asian": "mixed_or_multiple_white_and_asian",
    "Mixed or Multiple ethnic groups: White and Black African": "mixed_or_multiple_white_and_black_african",
    "Mixed or Multiple ethnic groups: White and Black Caribbean": "mixed_or_multiple_white_and_black_caribbean",
    
    "Other ethnic group: Any other ethnic group": "other_other",
    "Other ethnic group: Arab": "other_arab",
    
    "White: English, Welsh, Scottish, Northern Irish or British": "white_british",
    "White: Gypsy or Irish Traveller": "white_gypsy_or_irish",
    "White: Irish": "white_irish",
    "White: Other White": "white_other",
    "White: Roma": "white_roma"
}

# Rename columns using the dictionary
lsoa_ethincity_df_P.rename(columns=column_rename_map, inplace=True)

# Rename columns: lowercase, replace spaces with _, remove colons, and add '_count' suffix
lsoa_ethincity_df_P.columns = (
    lsoa_ethincity_df_P.columns
    .str.cat(['_count'] * len(lsoa_ethincity_df_P.columns))  # Append '_count' to each column
)

#rename columns
lsoa_ethincity_df_P.rename(columns = {'Lower tier local authorities Code_count':'lad22cd'},inplace = True)
# Display the updated DataFrame

lsoa_ethincity_df_P['total_ethnicity_pop'] = lsoa_ethincity_df_P.sum(axis=1,numeric_only=True)

lsoa_ethincity_df_P.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_37740\3682580362.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa_ethincity_df_P.drop(['Does not apply'], 1, inplace=True)


Ethnic group (20 categories),lad22cd,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop
0,E06000001,278,217,335,473,297,327,57,61,173,240,115,143,284,270,87761,37,170,1081,19,92338
1,E06000002,595,669,2804,2032,8990,3339,162,315,671,1110,650,570,2016,1452,114421,160,434,3212,320,143922
2,E06000003,158,208,175,336,283,182,48,35,240,460,215,270,211,321,131789,87,350,1148,14,136530
3,E06000004,236,690,1812,1439,4875,1823,130,250,725,1131,469,412,1106,558,177526,145,528,2595,143,196593
4,E06000005,759,308,1086,618,195,456,135,110,374,516,198,384,603,329,97320,434,342,3525,104,107796


## 2. Feature Engineering

In [7]:
# Create percentage columns for detailed ethnicity
total_ethnicity_pop = 'total_ethnicity_pop'
for col in lsoa_ethincity_df_P.columns[1:-1]:
    perc_col = col.replace('_count', '_perc')
    lsoa_ethincity_df_P[perc_col] = (lsoa_ethincity_df_P[col] / lsoa_ethincity_df_P[total_ethnicity_pop]) * 100

lsoa_ethincity_df_P.head()

Ethnic group (20 categories),lad22cd,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc
0,E06000001,278,217,335,473,297,327,57,61,173,240,115,143,284,270,87761,37,170,1081,19,92338,0.301068,0.235006,0.362798,0.512248,0.321644,0.354134,0.061730,0.066062,0.187355,0.259915,0.124542,0.154866,0.307566,0.292404,95.043211,0.040070,0.184106,1.170699,0.020577
1,E06000002,595,669,2804,2032,8990,3339,162,315,671,1110,650,570,2016,1452,114421,160,434,3212,320,143922,0.413418,0.464835,1.948278,1.411876,6.246439,2.320007,0.112561,0.218869,0.466225,0.771251,0.451634,0.396048,1.400759,1.008880,79.502091,0.111171,0.301552,2.231764,0.222343
2,E06000003,158,208,175,336,283,182,48,35,240,460,215,270,211,321,131789,87,350,1148,14,136530,0.115725,0.152347,0.128177,0.246100,0.207280,0.133304,0.035157,0.025635,0.175786,0.336922,0.157475,0.197759,0.154545,0.235113,96.527503,0.063722,0.256354,0.840841,0.010254
3,E06000004,236,690,1812,1439,4875,1823,130,250,725,1131,469,412,1106,558,177526,145,528,2595,143,196593,0.120045,0.350979,0.921701,0.731969,2.479742,0.927296,0.066126,0.127166,0.368782,0.575300,0.238564,0.209570,0.562584,0.283835,90.301282,0.073756,0.268575,1.319986,0.072739
4,E06000005,759,308,1086,618,195,456,135,110,374,516,198,384,603,329,97320,434,342,3525,104,107796,0.704108,0.285725,1.007459,0.573305,0.180897,0.423021,0.125237,0.102045,0.346952,0.478682,0.183680,0.356228,0.559390,0.305206,90.281643,0.402612,0.317266,3.270066,0.096479


In [8]:
# Feature Engineering - Aggregating Ethnicity Counts
ethnicity_groups = {
    "asian_count": [
        "asian_bangladeshi_count",
        "asian_chinese_count",
        "asian_indian_count",
        "asian_pakistani_count",
        "asian_other_count"
    ],
    "black_count": [
        "black_african_count",
        "black_caribbean_count",
        "black_other_count"
    ],
    "mixed_or_multiple_count": [
        "mixed_or_multiple_other_count",
        "mixed_or_multiple_white_and_asian_count",
        "mixed_or_multiple_white_and_black_african_count",
        "mixed_or_multiple_white_and_black_caribbean_count"
    ],
    "other_ethnic_count": [
        "other_other_count",
        "other_arab_count"
    ],
    "white_count": [
        "white_british_count",
        "white_gypsy_or_irish_count",
        "white_irish_count",
        "white_other_count",
        "white_roma_count"
    ]
}

# Compute aggregated ethnicity counts
for new_col, cols in ethnicity_groups.items():
    lsoa_ethincity_df_P[new_col] = lsoa_ethincity_df_P[cols].sum(axis=1)

# Display results
lsoa_ethincity_df_P.head()

Ethnic group (20 categories),lad22cd,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc,asian_count,black_count,mixed_or_multiple_count,other_ethnic_count,white_count
0,E06000001,278,217,335,473,297,327,57,61,173,240,115,143,284,270,87761,37,170,1081,19,92338,0.301068,0.235006,0.362798,0.512248,0.321644,0.354134,0.061730,0.066062,0.187355,0.259915,0.124542,0.154866,0.307566,0.292404,95.043211,0.040070,0.184106,1.170699,0.020577,1600,445,671,554,89068
1,E06000002,595,669,2804,2032,8990,3339,162,315,671,1110,650,570,2016,1452,114421,160,434,3212,320,143922,0.413418,0.464835,1.948278,1.411876,6.246439,2.320007,0.112561,0.218869,0.466225,0.771251,0.451634,0.396048,1.400759,1.008880,79.502091,0.111171,0.301552,2.231764,0.222343,15090,3816,3001,3468,118547
2,E06000003,158,208,175,336,283,182,48,35,240,460,215,270,211,321,131789,87,350,1148,14,136530,0.115725,0.152347,0.128177,0.246100,0.207280,0.133304,0.035157,0.025635,0.175786,0.336922,0.157475,0.197759,0.154545,0.235113,96.527503,0.063722,0.256354,0.840841,0.010254,1160,265,1185,532,133388
3,E06000004,236,690,1812,1439,4875,1823,130,250,725,1131,469,412,1106,558,177526,145,528,2595,143,196593,0.120045,0.350979,0.921701,0.731969,2.479742,0.927296,0.066126,0.127166,0.368782,0.575300,0.238564,0.209570,0.562584,0.283835,90.301282,0.073756,0.268575,1.319986,0.072739,9052,2203,2737,1664,180937
4,E06000005,759,308,1086,618,195,456,135,110,374,516,198,384,603,329,97320,434,342,3525,104,107796,0.704108,0.285725,1.007459,0.573305,0.180897,0.423021,0.125237,0.102045,0.346952,0.478682,0.183680,0.356228,0.559390,0.305206,90.281643,0.402612,0.317266,3.270066,0.096479,2966,701,1472,932,101725


In [9]:
# Create percentage columns for simplified ethnicity
total_ethnicity_pop = 'total_ethnicity_pop'
for col in lsoa_ethincity_df_P.columns[-5:]:
    perc_col = col.replace('_count', '_perc')
    lsoa_ethincity_df_P[perc_col] = (lsoa_ethincity_df_P[col] / lsoa_ethincity_df_P[total_ethnicity_pop]) * 100

lsoa_ethincity_df_P.head()

Ethnic group (20 categories),lad22cd,asian_bangladeshi_count,asian_chinese_count,asian_indian_count,asian_other_count,asian_pakistani_count,black_african_count,black_caribbean_count,black_other_count,mixed_or_multiple_other_count,mixed_or_multiple_white_and_asian_count,mixed_or_multiple_white_and_black_african_count,mixed_or_multiple_white_and_black_caribbean_count,other_other_count,other_arab_count,white_british_count,white_gypsy_or_irish_count,white_irish_count,white_other_count,white_roma_count,total_ethnicity_pop,asian_bangladeshi_perc,asian_chinese_perc,asian_indian_perc,asian_other_perc,asian_pakistani_perc,black_african_perc,black_caribbean_perc,black_other_perc,mixed_or_multiple_other_perc,mixed_or_multiple_white_and_asian_perc,mixed_or_multiple_white_and_black_african_perc,mixed_or_multiple_white_and_black_caribbean_perc,other_other_perc,other_arab_perc,white_british_perc,white_gypsy_or_irish_perc,white_irish_perc,white_other_perc,white_roma_perc,asian_count,black_count,mixed_or_multiple_count,other_ethnic_count,white_count,asian_perc,black_perc,mixed_or_multiple_perc,other_ethnic_perc,white_perc
0,E06000001,278,217,335,473,297,327,57,61,173,240,115,143,284,270,87761,37,170,1081,19,92338,0.301068,0.235006,0.362798,0.512248,0.321644,0.354134,0.061730,0.066062,0.187355,0.259915,0.124542,0.154866,0.307566,0.292404,95.043211,0.040070,0.184106,1.170699,0.020577,1600,445,671,554,89068,1.732764,0.481925,0.726678,0.599970,96.458663
1,E06000002,595,669,2804,2032,8990,3339,162,315,671,1110,650,570,2016,1452,114421,160,434,3212,320,143922,0.413418,0.464835,1.948278,1.411876,6.246439,2.320007,0.112561,0.218869,0.466225,0.771251,0.451634,0.396048,1.400759,1.008880,79.502091,0.111171,0.301552,2.231764,0.222343,15090,3816,3001,3468,118547,10.484846,2.651436,2.085157,2.409639,82.368922
2,E06000003,158,208,175,336,283,182,48,35,240,460,215,270,211,321,131789,87,350,1148,14,136530,0.115725,0.152347,0.128177,0.246100,0.207280,0.133304,0.035157,0.025635,0.175786,0.336922,0.157475,0.197759,0.154545,0.235113,96.527503,0.063722,0.256354,0.840841,0.010254,1160,265,1185,532,133388,0.849630,0.194097,0.867941,0.389658,97.698674
3,E06000004,236,690,1812,1439,4875,1823,130,250,725,1131,469,412,1106,558,177526,145,528,2595,143,196593,0.120045,0.350979,0.921701,0.731969,2.479742,0.927296,0.066126,0.127166,0.368782,0.575300,0.238564,0.209570,0.562584,0.283835,90.301282,0.073756,0.268575,1.319986,0.072739,9052,2203,2737,1664,180937,4.604437,1.120589,1.392216,0.846419,92.036339
4,E06000005,759,308,1086,618,195,456,135,110,374,516,198,384,603,329,97320,434,342,3525,104,107796,0.704108,0.285725,1.007459,0.573305,0.180897,0.423021,0.125237,0.102045,0.346952,0.478682,0.183680,0.356228,0.559390,0.305206,90.281643,0.402612,0.317266,3.270066,0.096479,2966,701,1472,932,101725,2.751494,0.650302,1.365542,0.864596,94.368066


## 4. Load GIS shapefile 

In [10]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,LAD22CD,LAD22NM,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
0,E06000001,Hartlepool,447160,531474,-1.27018,54.6761,ef33ae90-bd6b-43c8-aa11-b3308fb1b09d,"POLYGON ((447213.900 537036.104, 447228.798 53..."
1,E06000002,Middlesbrough,451141,516887,-1.21099,54.5447,408145f3-dcd6-455d-9dca-c1e11b5c9214,"POLYGON ((448489.897 522071.798, 448592.597 52..."
2,E06000003,Redcar and Cleveland,464361,519597,-1.00608,54.5675,4b8f6d25-4eb2-4ef4-8afb-5e20452921cd,"POLYGON ((455525.931 528406.654, 455724.632 52..."
3,E06000004,Stockton-on-Tees,444940,518183,-1.30664,54.5569,d978b075-04eb-4738-9611-fdc649760770,"POLYGON ((444157.002 527956.304, 444165.898 52..."
4,E06000005,Darlington,428029,515648,-1.56835,54.5353,eb7dbdf5-d3b5-4710-ab38-534d77fc4426,"POLYGON ((423496.602 524724.299, 423497.204 52..."


## 5. GIS data management

### Remove Rename fields

In [11]:
#Drop and rename fields
lsoa2021boundary_df.drop(['BNG_E', 'BNG_N', 'LONG','LAT','GlobalID',], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LAD22CD':'lad22cd','LAD22NM':'lad22nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_37740\111442805.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['BNG_E', 'BNG_N', 'LONG','LAT','GlobalID',], 1, inplace = True)


,lad22cd,lad22nm,geometry
0,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53..."
1,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52..."
2,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52..."
3,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52..."
4,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52..."


### Combine with boundary lookup table

#### LAD22 to REGION22

In [12]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_37740\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [13]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,lad22cd,lad22nm,geometry,rgn22cd,rgn22nm
0,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53...",E12000001,North East
1,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52...",E12000001,North East
2,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52...",E12000001,North East
3,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52...",E12000001,North East
4,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52...",E12000001,North East


#### LAD22 to SDS BOUNDARIES

In [19]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\0_Leads\2424_Place+Portrait\WORKING\Data_Prep\Raw Data\CSV\sds_ons_la_boundary_may2022.csv"

# Read the Excel file
sdslookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
sdslookup_df.drop(['lad22nm','bng_e','bng_n','long','lat','globalid','area_ha','id_original'],1,inplace = True)

# Display the first few rows
sdslookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_37740\4137896843.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  sdslookup_df.drop(['lad22nm','bng_e','bng_n','long','lat','globalid','area_ha','id_original'],1,inplace = True)


,fid,lad22cd,sds_boundary
0,1,E06000001,Tees Valley
1,2,E06000002,Tees Valley
2,3,E06000003,Tees Valley
3,4,E06000004,Tees Valley
4,5,E06000005,Tees Valley


In [20]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, sdslookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,lad22cd,lad22nm,geometry,rgn22cd,rgn22nm,fid_x,bng_n,sds_boundary_x,fid_y,sds_boundary_y
0,E06000001,Hartlepool,"POLYGON ((447213.900 537036.104, 447228.798 53...",E12000001,North East,1.0,531474.0,Tees Valley,1.0,Tees Valley
1,E06000002,Middlesbrough,"POLYGON ((448489.897 522071.798, 448592.597 52...",E12000001,North East,2.0,516887.0,Tees Valley,2.0,Tees Valley
2,E06000003,Redcar and Cleveland,"POLYGON ((455525.931 528406.654, 455724.632 52...",E12000001,North East,3.0,519597.0,Tees Valley,3.0,Tees Valley
3,E06000004,Stockton-on-Tees,"POLYGON ((444157.002 527956.304, 444165.898 52...",E12000001,North East,4.0,518183.0,Tees Valley,4.0,Tees Valley
4,E06000005,Darlington,"POLYGON ((423496.602 524724.299, 423497.204 52...",E12000001,North East,5.0,515648.0,Tees Valley,5.0,Tees Valley


### Update Area

In [ ]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

# 7. Combine Geometry and data

In [ ]:
census2021_ethnicity_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,lsoa_ethincity_df_P, how = 'left', on='lad22cd')
census2021_ethnicity_lsoa2021_gdb_df.head()

# 8. Final tweaks

In [ ]:
# Reorder columns in the combined dataframe

final_column_order = [
 'lad22cd',
 'lad22nm',
 'rgn22cd',
 'rgn22nm',
 'sds_boundary',
 'geometry', 
 'asian_count',
 'black_count',
 'mixed_or_multiple_count',
 'other_ethnic_count',
 'white_count',
 'asian_perc',
 'black_perc',
 'mixed_or_multiple_perc',
 'other_ethnic_perc',
 'white_perc',
 'asian_bangladeshi_count',
 'asian_chinese_count',
 'asian_indian_count',
 'asian_other_count',
 'asian_pakistani_count',
 'black_african_count',
 'black_caribbean_count',
 'black_other_count',
 'mixed_or_multiple_other_count',
 'mixed_or_multiple_white_and_asian_count',
 'mixed_or_multiple_white_and_black_african_count',
 'mixed_or_multiple_white_and_black_caribbean_count',
 'other_other_count',
 'other_arab_count',
 'white_british_count',
 'white_gypsy_or_irish_count',
 'white_irish_count',
 'white_other_count',
 'white_roma_count',
 'total_ethnicity_pop',
 'asian_bangladeshi_perc',
 'asian_chinese_perc',
 'asian_indian_perc',
 'asian_other_perc',
 'asian_pakistani_perc',
 'black_african_perc',
 'black_caribbean_perc',
 'black_other_perc',
 'mixed_or_multiple_other_perc',
 'mixed_or_multiple_white_and_asian_perc',
 'mixed_or_multiple_white_and_black_african_perc',
 'mixed_or_multiple_white_and_black_caribbean_perc',
 'other_other_perc',
 'other_arab_perc',
 'white_british_perc',
 'white_gypsy_or_irish_perc',
 'white_irish_perc',
 'white_other_perc',
 'white_roma_perc'
]

census2021_ethnicity_lsoa2021_gdb_df = census2021_ethnicity_lsoa2021_gdb_df[final_column_order]

census2021_ethnicity_lsoa2021_gdb_df.head()

# 8. Create dominant field

In [ ]:
def determine_dominant_group(row):
    age_columns = [
        'asian_perc',
        'black_perc',
        'mixed_or_multiple_perc',
        'other_ethnic_perc',
        'white_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - (threshold_value*2)
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_ethnicity_lsoa2021_gdb_df['dominant_ethnic_group'] = census2021_ethnicity_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_ethnicity_lsoa2021_gdb_df.head()

In [ ]:
def determine_dominant_group(row):
    age_columns = [
 'asian_bangladeshi_perc',
 'asian_chinese_perc',
 'asian_indian_perc',
 'asian_other_perc',
 'asian_pakistani_perc',
 'black_african_perc',
 'black_caribbean_perc',
 'black_other_perc',
 'mixed_or_multiple_other_perc',
 'mixed_or_multiple_white_and_asian_perc',
 'mixed_or_multiple_white_and_black_african_perc',
 'mixed_or_multiple_white_and_black_caribbean_perc',
 'other_other_perc',
 'other_arab_perc',
 'white_british_perc',
 'white_gypsy_or_irish_perc',
 'white_irish_perc',
 'white_other_perc',
 'white_roma_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - (threshold_value)
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_ethnicity_lsoa2021_gdb_df['dominant_detailed_ethnic_group'] = census2021_ethnicity_lsoa2021_gdb_df.apply(determine_dominant_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_ethnicity_lsoa2021_gdb_df.head()

# 9. Upload to geodatabase

In [ ]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "place_plus"
table_name = "census2021_lad22_ethnicity"  # Desired table name
primary_key_column = "lad22cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [ ]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_ethnicity_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_ethnicity_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [ ]:
# Publish the GeoDataFrame to PostGIS
census2021_ethnicity_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'MULTIPOLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")